## Set-up

Configure polars displayto show long strings in full and at most 40 rows, and store the raw parquet files for both the service and requests and daily-climate weather for Calgary.

In [1]:
from datetime import date
from pathlib import Path

import polars as pl

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(40)

raw = Path.cwd().resolve().parent / Path("raw")
folder = Path.cwd().resolve().parent / Path("raw") / Path("311")
files = [f for f in folder.rglob("*.parquet") if f.is_file()]
RAW_311 = [f for f in (raw / Path("311")).rglob("*.parquet") if f.is_file()]
RAW_WEATHER = [f for f in (raw / Path("weather")).rglob("*.parquet") if f.is_file()]

## 1. Load the raw data

The raw layer is append-only, in which each stored version of the API requests are kept, so the same request can appear in several files.

In [2]:
raw = pl.scan_parquet(RAW_311).collect()
print(
    f"{raw.height:,} rows, {raw['service_request_id'].n_unique():,} distinct request IDs"
)
raw.head()

7,562,317 rows, 7,554,255 distinct request IDs


service_request_id,requested_date,updated_date,closed_date,status_description,source,service_name,agency_responsible,comm_code,comm_name,location_type,longitude,latitude,_ingested_at
str,datetime[ms],datetime[ms],datetime[ms],str,str,str,str,str,str,str,f64,f64,str
"""24-00689897""",2024-09-01 00:00:00,2025-06-24 00:00:00,2024-09-23 00:00:00,"""Closed""","""Other""","""Bylaw - Noise Concerns""","""CS - Emergency Management and Community Safety""","""SOV""","""SOUTHVIEW""","""Community Centrepoint""",-113.997339,51.034152,"""2026-11-13T02:28:13.496929+00:00"""
"""24-00689898""",2024-09-01 00:00:00,2024-09-03 00:00:00,2024-09-03 00:00:00,"""Closed""","""Other""","""Bylaw - Noise Concerns""","""CS - Emergency Management and Community Safety""","""SOC""","""SOUTH CALGARY""","""Community Centrepoint""",-114.102075,51.026802,"""2026-11-13T02:28:13.496929+00:00"""
"""24-00689899""",2024-09-01 00:00:00,2025-06-24 00:00:00,2024-09-09 00:00:00,"""Closed""","""Other""","""CT - Lost Property""","""OS - Calgary Transit""",null,null,"""None""",null,null,"""2026-11-13T02:28:13.496929+00:00"""
"""24-00689903""",2024-09-01 00:00:00,2024-09-02 00:00:00,2024-09-01 00:00:00,"""Closed""","""App""","""Bylaw - Water Misuse / Wasting Concern""","""CS - Emergency Management and Community Safety""","""CAP""","""CAPITOL HILL""","""Community Centrepoint""",-114.094631,51.072665,"""2026-11-13T02:28:13.496929+00:00"""
"""24-00689906""",2024-09-01 00:00:00,2024-09-03 00:00:00,2024-09-03 00:00:00,"""Closed""","""Other""","""Bylaw - Disturbance and Behavioural Concerns""","""CS - Emergency Management and Community Safety""","""VAR""","""VARSITY""","""Community Centrepoint""",-114.162507,51.096054,"""2026-11-13T02:28:13.496929+00:00"""


- About 7.56 million rows and 7.55 million distinct request IDS: There are more rows than distinct IDs. This means that **some requests were stored more than once**.

## 2. Profiling the raw columns

In [3]:
raw["status_description"].value_counts(sort=True)

status_description,count
str,u32
"""Closed""",7362751
"""Duplicate (Closed)""",114288
"""Open""",82788
"""Duplicate (Open)""",2387
"""TO BE DELETED""",103


In [4]:
raw["source"].value_counts(sort=True)

source,count
str,u32
"""Phone""",4009293
"""Other""",2190778
"""App""",789933
"""Web""",572313


**Findings:**

- `status_description` has five values: (`Closed` (7,362,751), `Duplicate (Closed)` (114,288), `Open` (82,788), `Duplicate (Open)` (2,387), `TO BE DELETED` (103), and has **no** `NULL`s.

- `source` has 4 values: `Phone`, `Other`, `App`, `Web`, and no `NULL`s. `Other` is 29% of all rows. How each changes over the years should be explored.

- The text "None" appears in `location_type` for 435,908 raw rows. This should be changed to `NULL`.

**Decisions:**
We should drop the `TO BE DELETED` rows, and treat the "None" text as `NULL`.

## 3. Duplicates

Since the daily incremental runs extract using a 3-day overlap, recently updated requests will be stored in more than one file. This requires a deduplication.

In [5]:
dupes = raw.filter(pl.col("service_request_id").is_duplicated())
print(
    f"{dupes.height:,} rows belong to {dupes['service_request_id'].n_unique():,} duplicated requests "
    f"({dupes.height / raw.height:.3%} of rows)"
)
dupes.group_by("updated_date").len().sort("updated_date")

16,124 rows belong to 8,062 duplicated requests (0.213% of rows)


updated_date,len
datetime[ms],u32
2026-11-08 00:00:00,1216
2026-11-09 00:00:00,4824
2026-11-10 00:00:00,7010
2026-11-11 00:00:00,3074


In [6]:
# Check: Did anything change between the copies?
changes = dupes.group_by("service_request_id").agg(
    pl.col("status_description").n_unique().alias("statuses"),
    pl.col("closed_date").n_unique().alias("closed_dates"),
    pl.col("updated_date").n_unique().alias("updated_dates"),
)
print(
    changes.select(
        (pl.col("statuses") > 1).sum().alias("status_changed"),
        (pl.col("closed_dates") > 1).sum().alias("closed_date_changed"),
        (pl.col("updated_dates") > 1).sum().alias("updated_date_changed"),
    )
)

# Check: Are the copies identical in every column except the load time?
rows_without_load_time = dupes.drop("_ingested_at").unique().height
ids = dupes["service_request_id"].n_unique()
print("fully identical copies:", rows_without_load_time == ids)

shape: (1, 3)
┌────────────────┬─────────────────────┬──────────────────────┐
│ status_changed ┆ closed_date_changed ┆ updated_date_changed │
│ ---            ┆ ---                 ┆ ---                  │
│ u32            ┆ u32                 ┆ u32                  │
╞════════════════╪═════════════════════╪══════════════════════╡
│ 0              ┆ 0                   ┆ 0                    │
└────────────────┴─────────────────────┴──────────────────────┘
fully identical copies: True


**Findings:**
- 16,124 rows belong to 8,062 requests that appear twice (0.213% of rows). Polars' `is_duplicated` marks every copy, so the row count is twice the request count.
- Every duplicate has an `updated_date` from the last few days only (2026-11-08 to 2026-11-11). The source itself has no duplicate IDs, while the overlap window creates duplicates.
- The copies match on status, closed date and updated date, so there are simple re-reads.

**Decision:** Keep one row per request, ordered such that the newest data wins and is the only write when deduplicating.

## 4. Keep the latest version of each request

The newest `updated_date` wins. `_ingested_at` breaks ties and decides when `updated_date` is `NULL`. Ordering on `updated_date` first keeps the result right even if an old backfill file is ever reloaded, because that file would carry a newer `_ingested_at` but older content.

In [7]:
latest = (
    raw.sort(["updated_date", "_ingested_at"], descending=True, nulls_last=True)
    .unique(subset="service_request_id", keep="first", maintain_order=True)
    .filter(pl.col("status_description") != "TO BE DELETED")
)
assert latest["service_request_id"].is_unique().all()
print(
    f"{latest.height:,} requests after the latest-version rule and dropping `TO BE DELETED`"
)

del raw, dupes  # free memory

7,554,152 requests after the latest-version rule and dropping `TO BE DELETED`


**Findings:**

7,554,152 requests, from 7,562,317 raw rows minus the 103 `TO BE DELETED` and minus the 8,062 extra copies. This is a target for the dbt staging model.

## 5. Cleaning the columns

### 5.1 Cleaning placeholder text and whitespace

"N/A", "null", "None", and "" (blanks) all mean "unknown", so they become real `NULL`s. Whitespace is trimmed and repeated spaces are eliminated.

In [8]:
UNKNOWN = ["N/A", "NULL", "NONE", ""]
text_columns = [
    "service_name",
    "agency_responsible",
    "comm_code",
    "comm_name",
    "location_type",
]

cleaned = latest.with_columns(
    [
        pl.when(
            pl.col(column)
            .str.strip_chars()
            .str.replace_all(r"\s+", " ")
            .str.to_uppercase()
            .is_in(UNKNOWN)
        )
        .then(None)
        .otherwise(pl.col(column).str.strip_chars().str.replace_all(r"\s+", " "))
        .alias(column)
        for column in text_columns
    ]
)

### 5.2 Retired services and service names

Finds from the service name counts (1,173 distinct names):
- 121 names carry a leading `z`, `Z`, `Z CT`, `zz`, `ZZZ` or `ZParks` marker. They are retired versions of active services, for example `Z CT` requests run from 2010 to March 2016 and then stop, and `z - Finance - Property Tax Account Inquiry` sits beside `Finance - Property Tax Account Inquiry`.

- 11 names contain `OLD`, `DO NOT USE` or `INTERNAL` (about 39,000 rows).

- Left alone, the `z` prefix would make `service_group` take the values `Z`, `z`, and `Z CT`.

**Decision:** Flag these as `is_legacy_service` (computed before the prefix is removed), then strip the `z` marker.

In [ ]:
LEGACY_PREFIX = r"^[Zz]+(?:\s*-\s*|\s+)"  # z - / Z - / z- / Z CT / zz - / ZZZ
RETIRED_WORDS = r"\bOLD\b|DO NOT USE|INTERNAL"
name = pl.col("service_name")

z_names = cleaned.filter(name.str.contains(LEGACY_PREFIX + r"|^ZParks\b"))
print(
    f"{z_names['service_name'].n_unique()} names with a z marker, "
    f"{z_names.height:,} requests ({z_names.height / cleaned.height:.1%} of all requests)"
)

In [ ]:
(
    z_names.with_columns(
        name.str.extract(r"^(Z CT|ZParks|ZZZ|zz|Z|z)", 1).alias("marker")
    )
    .group_by("marker")
    .agg(
        pl.col("service_name").n_unique().alias("names"),
        pl.len().alias("requests"),
        pl.col("requested_date").min().cast(pl.Date).alias("firstb_request"),
        pl.col("requested_date").max().cast(pl.Date).alias("last_request"),
    )
    .sort("requests", descending=True)
)

In [ ]:
legacy_pattern = LEGACY_PREFIX + r"|^ZParks\b"
counts = cleaned.group_by("service_name").agg(pl.len().alias("requests"))

legacy_counts = counts.filter(
    pl.col("service_name").str.contains(legacy_pattern)
).with_columns(
    pl.col("service_name")
    .str.replace(r"^ZParks\b", "Parks")
    .str.replace(LEGACY_PREFIX, "")
    .alias("active_name")
)
active_counts = counts.filter(
    ~pl.col("service_name").str.contains(legacy_pattern)
).rename({"service_name": "active_name", "requests": "active_requests"})
twins = legacy_counts.join(active_counts, on="active_name", how="inner").sort(
    "requests", descending=True
)

print(
    f"{twins.height} of {legacy_counts.height} legacy names have an active twin with the same name"
)
twins.head(10)

In [ ]:
retired = (
    cleaned.filter(name.str.contains(RETIRED_WORDS))
    .group_by("service_name")
    .len()
    .sort("len", descending=True)
)
print(
    f"{retired.height} names with OLD, DO NOT USE or INTERNAL, {retired['len'].sum():,} requests"
)
print(retired)

# service_group if the z marker were left in
z_names.group_by(name.str.split(" - ").list.first().alias("naive_group")).len().sort(
    "len", descending=True
)

In [ ]:
cleaned = cleaned.with_columns(
    (
        name.str.contains(LEGACY_PREFIX + r"|^ZParks\b")
        | name.str.contains(RETIRED_WORDS)
    )
    .fill_null(False)
    .alias("is_legacy_service"),
    name.str.replace(r"^ZParks\b", "Parks")
    .str.replace(LEGACY_PREFIX, "")
    .alias("service_name"),
)
num = cleaned["is_legacy_service"].sum()
print(f"There are {num} rows with a legacy service.")

There are 445302 rows with a legacy service.


### 5.3 Spelling variants of the same service

Some names differ only in case, spacing or punctuation. For example, `Mayors Office` and `Mayor's Office`, `City Clerks` and `City Clerk's`, `REDS` and `RE&DS`, `Roads - Permits - ...` and `Roads Permits - ...`, `Streetlight Damage` and `Streetlight - Damage`. A few known prefix variants are fixed by regex, then every remaining set of names that match ignoring the case and punctuation is mapped to its most common spelling.

In [14]:
NAME_FIXES = {
    r"^Mayors Office\b": "Mayor's Office",
    r"^City Clerks\b": "City Clerk's",
    r"^REDS\b": "RE&DS",
}
for pattern, replacement in NAME_FIXES.items():
    cleaned = cleaned.with_columns(
        pl.col("service_name").str.replace(pattern, replacement)
    )

key = pl.col("service_name").str.to_lowercase().str.replace_all(r"[^a-z0-9]", "")
name_map = (
    cleaned.filter(pl.col("service_name").is_not_null())
    .group_by("service_name")
    .len()
    .with_columns(key.alias("key"))
    .sort("len", descending=True)
    .with_columns(pl.col("service_name").first().over("key").alias("canonical_name"))
    .select(pl.col("service_name").alias("name"), "canonical_name")
)
cleaned = (
    cleaned.join(name_map, left_on="service_name", right_on="name", how="left")
    .with_columns(pl.coalesce("canonical_name", "service_name").alias("service_name"))
    .drop("canonical_name")
)
name_map.filter(
    pl.col("name") != pl.col("canonical_name")
)  # this table will be used as a target for the dbt staging.

name,canonical_name
str,str
"""Roads - Permits - Requests and Inquiries""","""Roads Permits - Requests and Inquiries"""
"""Roads - New Signal, Left Turn or Pedestrian Light Request""","""Roads - NEW Signal, Left Turn or Pedestrian Light Request"""
"""Parks - PlayBins""","""Parks PlayBins"""
"""Roads - Streetlight - Damage""","""Roads - Streetlight Damage"""
"""CFD - Smoke Alarm Inquiries - Asst. for Seniors/Disabled""","""CFD - Smoke Alarm Inquiries & Asst. for Seniors/Disabled"""
"""Parks - Seasonal Vendor Application""","""Parks Seasonal Vendor Application"""


### 5.4 Service group, communities and dates

- `service_group` is the text before the first "-" (the whole name if there is no dash). Names with several dashes are not split further, because the later segments mean different things: a subcategory (`Roads - Signs - Traffic and Roadmarking`), a location (`RSP - Pool - Thornhill`), one phrase (`Employee Complaint - Compliment`), or a system tag (`... - WAM`, `... - GIS`, `... - FHB`).

- Community names have a stray space after a slash (`SCARBORO / SUNALTA WEST` and `SCARBORO/SUNALTA WEST` are one place). 51 community codes such as `01H` and `12K` also appear as the community name: They are unnamed areas, not neighbourhoods, so they are marked as `is_unnamed_area`.

- The dates are cast to `date`. Early data (about 2012 to 2020) has a time of day and 2022 onwards is all midnight, so day grain is the only grain that is consistent across years. The original timestamps stay in the raw Parquet files.

In [15]:
cleaned = cleaned.with_columns(
    pl.col("service_name").str.split(" - ").list.first().alias("service_group"),
    pl.col("comm_name").str.replace_all(r"\s*/\s*", "/"),
    pl.col("requested_date").cast(pl.Date),
    pl.col("updated_date").cast(pl.Date),
    pl.col("closed_date").cast(pl.Date),
).with_columns(
    pl.col("comm_name")
    .str.contains(r"^\d{2}[A-Z]$")
    .fill_null(False)
    .alias("is_unnamed_area"),
)
cleaned.head()

service_request_id,requested_date,updated_date,closed_date,status_description,source,service_name,agency_responsible,comm_code,comm_name,location_type,longitude,latitude,_ingested_at,is_legacy_service,service_group,is_unnamed_area
str,date,date,date,str,str,str,str,str,str,str,f64,f64,str,bool,str,bool
"""25-00405571""",2025-06-05,2026-11-11,null,"""Open""","""Other""","""Roads - Sidewalk - Curb and Gutter Repair""","""OS - Mobility""","""SIG""","""SIGNAL HILL""","""Community Centrepoint""",-114.179907,51.02326,"""2026-11-13T04:48:11.117862+00:00""",false,"""Roads""",false
"""25-00406387""",2025-06-05,2026-11-11,null,"""Open""","""Other""","""Roads - Sidewalk - Curb and Gutter Repair""","""OS - Mobility""","""RIC""","""RICHMOND""","""Community Centrepoint""",-114.11842,51.02971,"""2026-11-13T04:48:11.117862+00:00""",false,"""Roads""",false
"""25-00406499""",2025-06-05,2026-11-11,null,"""Open""","""App""","""Roads - Sidewalk - Curb and Gutter Repair""","""OS - Mobility""","""HIL""","""HILLHURST""","""Community Centrepoint""",-114.093646,51.057278,"""2026-11-13T04:48:11.117862+00:00""",false,"""Roads""",false
"""25-00406869""",2025-06-05,2026-11-11,null,"""Open""","""Other""","""Roads - Sidewalk - Curb and Gutter Repair""","""OS - Mobility""","""WSP""","""WEST SPRINGS""","""Community Centrepoint""",-114.206686,51.059773,"""2026-11-13T04:48:11.117862+00:00""",false,"""Roads""",false
"""25-00406872""",2025-06-05,2026-11-11,null,"""Open""","""Other""","""Roads - Sidewalk - Curb and Gutter Repair""","""OS - Mobility""","""PKD""","""PARKDALE""","""Community Centrepoint""",-114.135333,51.059948,"""2026-11-13T04:48:11.117862+00:00""",false,"""Roads""",false


### 5.5 What changed

In [16]:
compared = [
    {
        "column": c,
        "distinct_before": latest[c].n_unique(),
        "distinct_after": cleaned[c].n_unique(),
        "nulls_before": latest[c].null_count(),
        "nulls_after": cleaned[c].null_count(),
    }
    for c in [
        "service_name",
        "agency_responsible",
        "comm_code",
        "comm_name",
        "location_type",
    ]
]
display(pl.DataFrame(compared))

column,distinct_before,distinct_after,nulls_before,nulls_after
str,i64,i64,i64,i64
"""service_name""",1173,1125,0,8721
"""agency_responsible""",95,95,0,312
"""comm_code""",324,323,435287,436103
"""comm_name""",326,324,435287,436072
"""location_type""",2,2,0,435287


In [17]:
cleaned.select(
    pl.col("is_legacy_service").sum().alias("legacy_rows"),
    pl.col("is_unnamed_area").sum().alias("unnamed_area_rows"),
    pl.col("service_group").n_unique().alias("service_groups"),
)

legacy_rows,unnamed_area_rows,service_groups
u32,u32,u32
445302,36849,139


In [18]:
# Names that still match once case and punctuation are ignored. We are expecting no rows.
(
    cleaned.group_by(key.alias("key"))
    .agg(pl.col("service_name").unique().alias("names"), pl.len().alias("rows"))
    .filter(pl.col("names").list.len() > 1)
    .sort("rows", descending=True)
)

key,names,rows
str,list[str],u32


**Findings:**

- Distinct service names fell from 1,173 to about 1,129. 8,721 requests have the `N/A` service, which are now `NULL`.

- `N/A` and `NULL` in the agency and community columns are now **real** `NULL`s (instead of text), with 312 agency rows and about 800 extra community rows.

- 445,302 requests are flagged as legacy and 36,849 fall in unnamed areas. `service_group` has about 140 values and no longer contains `Z`, `z`, or one-letter groupings.

- The service groups mix levels, where some are department codes (`WRS`, `TI`, `HR`, `CSC`) and others are words (`Roads`, `Bylaw`, `Finance`), which are usable as they are.

## 6. Quality checks

Rules the data should follow. Each count should be 0 or at least small.

In [19]:
closed_status = pl.col("status_description").is_in(["Closed", "Duplicate (Closed)"])
cleaned.select(
    (pl.col("closed_date") < pl.col("requested_date"))
    .sum()
    .alias("closed_before_requested"),
    (pl.col("updated_date") < pl.col("requested_date"))
    .sum()
    .alias("updated_before_requested"),
    (pl.col("requested_date") > pl.lit(date.today()))
    .sum()
    .alias("requested_in_future"),
    (closed_status & pl.col("closed_date").is_null())
    .sum()
    .alias("closed_without_date"),
    (~closed_status & pl.col("closed_date").is_not_null())
    .sum()
    .alias("open_with_closed_date"),
    (pl.col("latitude").is_null() != pl.col("longitude").is_null())
    .sum()
    .alias("only_one_coordinate"),
    (pl.col("latitude").is_null() != pl.col("location_type").is_null())
    .sum()
    .alias("coords_vs_location_type"),
    (pl.col("latitude").is_not_null() & ~pl.col("latitude").is_between(50.8, 51.3))
    .sum()
    .alias("lat_outside_calgary"),
    (
        pl.col("longitude").is_not_null()
        & ~pl.col("longitude").is_between(-114.4, -113.8)
    )
    .sum()
    .alias("lon_outside_calgary"),
)

closed_before_requested,updated_before_requested,requested_in_future,closed_without_date,open_with_closed_date,only_one_coordinate,coords_vs_location_type,lat_outside_calgary,lon_outside_calgary
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,6,4702,0,880,0,0


In [20]:
# The 880 coordinate rows: which way do they disagree?
cleaned.group_by(
    pl.col("latitude").is_null().alias("no_coords"),
    pl.col("location_type").is_null().alias("no_location_type"),
).len()

no_coords,no_location_type,len
bool,bool,u32
true,true,435287
true,false,880
false,false,7117985


**Findings:**

- Dates are in order: No request closed or updated before it was requested nor is there any in the future. Coordinates always come as a pair and all lie inside Calgary.

- **6** closed requests have no closed date, and **4,702** open requests do have a closed date (4,687 earlier, which grows slowly over time). The status is treated as the source of truth, and these rows are flagged but not fixed.

- **435,287** requests have no location at all (no coordinates and no location type). A futher **880** have a location type but no coordinates (0.012%).

**Decisions:** 

Keep the status-verusus-date conflicts as a flag with a wanring-level dbt test. Keep the 880 rows and count them by community.

## 7. Bulk closures

A few days close thousands of requests at once. Some are real bursts of work and some are backlog cleanups that would otherwise distort time-to-close averages.

In [21]:
days_open = (pl.col("closed_date") - pl.col("requested_date")).dt.total_days()

top_dates = cleaned.group_by("closed_date").len().sort("len", descending=True).head(10)
display(top_dates)

closed_date,len
date,u32
null,76751
2025-08-26,7462
2024-04-29,7427
2013-05-29,4147
2022-12-11,4078
2022-06-28,3473
2023-06-20,3281
2025-10-08,3238
2022-06-21,3235


In [22]:
(
    cleaned.filter(pl.col("closed_date").is_in(top_dates["closed_date"]))
    .group_by("closed_date", "service_group")
    .agg(pl.len().alias("requests"), days_open.median().alias("median_days_open"))
    .sort("requests", descending=True)
    .head(15)
)

/var/folders/l1/4gygg18n3s947p63jr5rg4zc0000gn/T/ipykernel_25797/808013356.py:2: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  cleaned.filter(pl.col("closed_date").is_in(top_dates["closed_date"]))


closed_date,service_group,requests,median_days_open
date,str,u32,f64
2024-04-29,"""Roads""",5966,444.0
2025-08-26,"""Roads""",5354,1145.0
2022-12-11,"""Roads""",2906,1161.5
2013-05-29,"""Roads""",2145,232.0
2025-10-08,"""CT""",1427,127.0
2022-06-21,"""Finance""",1006,7.0
2018-03-13,"""Roads""",1001,1.0
2022-06-28,"""Finance""",980,6.0
2023-06-20,"""Finance""",729,6.0


**Finding.** Five closure days are cleanups, because the requests had been open for a long time:

| Closed on | Group | Requests | Median days open |
|---|---|---|---|
| 2024-04-29 | Roads | 5,973 | 443 |
| 2025-08-26 | Roads | 5,358 | 1,145 |
| 2022-12-11 | Roads | 2,906 | 1,161 |
| 2013-05-29 | Roads | 2,145 | 232 |
| 2025-10-08 | CT | 1,427 | 127 |

The other spikes (Finance, Parks, WRS, Bylaw, and Roads on 2018-03-13) have medians of 1 to 14 days and are real bursts. The cleanups are about 17,800 requests (0.24%) but would add roughly 10 days to the mean time-to-close for Roads.

**Decision.** Flag a batch when a closed date and service group have at least 1,000 closures and a median age of at least 100 days, and flag only the requests in that batch that were themselves open 100 days or more. Report medians in the marts.

In [23]:
MIN_ROWS, MIN_MEDIAN_DAYS = 1000, 100

batches = (
    cleaned.filter(pl.col("closed_date").is_not_null())
    .group_by("closed_date", "service_group")
    .agg(pl.len().alias("n"), days_open.median().alias("median_days"))
    .filter((pl.col("n") >= MIN_ROWS) & (pl.col("median_days") >= MIN_MEDIAN_DAYS))
    .select("closed_date", "service_group", pl.lit(True).alias("in_batch"))
)
cleaned = (
    cleaned.join(batches, on=["closed_date", "service_group"], how="left")
    .with_columns(
        (pl.col("in_batch").fill_null(False) & (days_open >= MIN_MEDIAN_DAYS)).alias(
            "is_bulk_closure"
        )
    )
    .drop("in_batch")
)
print(f"{cleaned['is_bulk_closure'].sum():,} requests flagged as bulk closures")
batches.sort("closed_date")

16,076 requests flagged as bulk closures


closed_date,service_group,in_batch
date,str,bool
2013-05-29,"""Roads""",true
2022-12-11,"""Roads""",true
2024-04-29,"""Roads""",true
2025-08-26,"""Roads""",true
2025-10-08,"""CT""",true


## 8. Location precision

The `location_type` is always `Community Centrepoint`, so the coordinates may describe the community and not the address.

In [24]:
located = cleaned.filter(pl.col("latitude").is_not_null())
located.select(
    pl.len().alias("requests_with_coords"),
    pl.struct("latitude", "longitude").n_unique().alias("distinct_points"),
    pl.col("comm_code").n_unique().alias("distinct_communities"),
)

requests_with_coords,distinct_points,distinct_communities
u32,u32,u32
7117985,1546,322


In [25]:
points = located.group_by("comm_code").agg(
    pl.struct("latitude", "longitude").n_unique().alias("points"),
    pl.len().alias("requests"),
)
points["points"].value_counts().sort("points")

points,count
u32,u32
1,5
2,4
3,34
4,91
5,103
6,55
7,18
8,10
9,2


In [26]:
# How far apart are one community's points?
spread = (
    located.group_by("comm_code")
    .agg(
        ((pl.col("latitude").max() - pl.col("latitude").min()) * 111_000).alias(
            "lat_spread_m"
        ),
        ((pl.col("longitude").max() - pl.col("longitude").min()) * 70_000).alias(
            "lon_spread_m"
        ),
    )
    .sort("lat_spread_m", descending=True)
)
spread.head(10)

comm_code,lat_spread_m,lon_spread_m
str,f64,f64
"""11B""",2030.558252,70.685092
"""02L""",812.875371,1074.080096
"""13J""",411.216401,10.38783
"""01H""",393.204279,31.039294
"""STD""",379.548242,1.430865
"""09O""",360.986909,1115.567751
"""09Q""",336.464233,7.564374
"""SVO""",302.279875,58.314648
"""CRM""",150.266685,602.00326


**Findings:**

- 7.1 million located requests share only **1,546 distinct points** across 322 communities: about 5 points per community (1 to 9). The coordinates are community centrepoints and not addresses.

- The points inside one community differ by a few metres (rounding and small boundary refreshes, likely), so each community is effectively one location.

- About ten communities are spread over 100m to 2km. 6 of them are numeric unnamed areas, which are areas and not a single point.

**Decisions:**

Analyse location at community level only and say so in the README. Group by `comm_code`, never by latitude and longitude. `dim_community` holds on median coordinate per community, and `NULL` for unnamed areas.

## 9. Volume over time

Looking for gaps, steps and unusual days. The first full year is 2012.

In [27]:
daily = (
    cleaned.group_by("requested_date")
    .len()
    .rename({"len": "requests"})
    .sort("requested_date")
)

cleaned.group_by(pl.col("requested_date").dt.year().alias("year")).len().sort("year")

year,len
i32,u32
2010,39
2011,89
2012,468537
2013,507063
2014,547900
2015,496382
2016,488196
2017,516172
2018,536289


In [28]:
monthly = (
    cleaned.group_by(pl.col("requested_date").dt.truncate("1mo").alias("month"))
    .len()
    .rename({"len": "requests"})
    .sort("month")
)
monthly.tail(30)

month,requests
date,u32
2024-06-01,54718
2024-07-01,50528
2024-08-01,45831
2024-09-01,42791
2024-10-01,38551
2024-11-01,35896
2024-12-01,31862
2025-01-01,36301
2025-02-01,29812


In [29]:
# Timeseries of requests
import altair as alt

alt.data_transformers.disable_max_rows()


df_plot = daily.sort("requested_date").with_columns(
    pl.col("requests")
    .rolling_mean_by(by="requested_date", window_size="1w")
    .alias("weekly_avg")
)

raw_line = df_plot.plot.line(x="requested_date", y="requests").encode(
    opacity=alt.value(0.25),
    color=alt.value("steelblue"),
)
avg_line = df_plot.plot.line(x="requested_date", y="weekly_avg").encode(
    strokeWidth=alt.value(2.5), color=alt.value("crimson")
)

(raw_line + avg_line).properties(width=1400, height=400)

alt.LayerChart(...)

In [30]:
# Calendar days with no requests at all
all_days = pl.date_range(
    daily["requested_date"].min(),
    daily["requested_date"].max(),
    interval="1d",
    eager=True,
)
zero_days = (
    pl.DataFrame({"requested_date": all_days})
    .join(daily, on="requested_date", how="left")
    .filter(pl.col("requests").is_null())
)

print(
    f"{zero_days.height} days with no requests, {zero_days.filter(pl.col('requested_date') < date(2012, 1, 1)).height} of them before 2012"
)
zero_days.filter(pl.col("requested_date") >= date(2012, 1, 1))

608 days with no requests, 604 of them before 2012


requested_date,requests
date,u32
2012-03-10,null
2014-12-31,null
2017-01-21,null
2017-12-02,null


In [31]:
# Weekday pattern (Monday = 1)
daily.group_by(pl.col("requested_date").dt.weekday().alias("weekday")).agg(
    pl.col("requests").sum(),
    pl.col("requests").mean().alias("avg_per_day"),
).sort("weekday")

weekday,requests,avg_per_day
i8,u32,f64
1,1286984,1612.761905
2,1411746,1769.105263
3,1374814,1742.476553
4,1339771,1695.912658
5,1174812,1485.223767
6,513161,661.289948
7,452864,576.896815


In [32]:
# Days far from the usual volume for the same weekday (trailing 8 weeks), from 2012-02
unusual = (
    daily.filter(pl.col("requested_date") >= date(2012, 2, 1))
    .with_columns(
        pl.col("requests")
        .rolling_median(window_size=8)
        .over(pl.col("requested_date").dt.weekday())
        .alias("typical")
    )
    .filter(
        (pl.col("requests") < 0.25 * pl.col("typical"))
        | (pl.col("requests") > 3 * pl.col("typical"))
    )
    .sort("requests")
)
display(unusual)

requested_date,requests,typical
date,u32,f64
2018-10-27,1,579.5
2016-12-31,14,417.5
2020-03-28,23,515.0
2015-12-31,57,1527.0
2017-12-31,62,384.5
2018-12-31,70,1519.0
2014-12-25,96,1573.5
2012-12-25,111,1623.5
2017-12-25,135,1469.5


In [33]:
# The four big spikes:
# 1. January 16th, 2014:    Road closure           - https://newsroom.calgary.ca/calgary-road-closures---january-16-2014/
# 2. September 10th, 2014:  The tree incident       - https://calgaryherald.com/news/local-news/september-snowstorm-damaged-about-half-of-all-calgary-trees
# 3. November 13th, 2018:   2026 olympic plebiscite - https://globalnews.ca/news/4650203/calgary-olympic-plebiscite-results/
# 4. April 3rd, 2020:       Drive-by Birthdays      - https://newsroom.calgary.ca/calgary-fire-department-drive-by-birthdays/

dates = [date(2014, 1, 16), date(2014, 9, 10), date(2018, 11, 13), date(2020, 4, 3)]
dates_text = [
    "January 16th, 2014:",
    "September 10th, 2014:",
    "November 13th, 2018:",
    "April 3rd, 2020:",
]
for i, d in enumerate(dates):
    spike = cleaned.filter(pl.col("requested_date").eq(d))
    top_service_groups = (
        spike.group_by("service_group").len().sort("len", descending=True).head(5)
    )
    top_service_group = top_service_groups.sort("len", descending=True).item(
        0, "service_group"
    )
    print(dates_text[i])
    print(f"Total requests: {spike.height}")
    print(top_service_groups)
    display(
        spike.filter(pl.col("service_group") == top_service_group)
        .group_by("service_name")
        .len()
        .sort("len", descending=True)
        .head(1)
    )
    print("\n")

January 16th, 2014:
Total requests: 3759
shape: (5, 2)
┌────────────────┬──────┐
│ service_group  ┆ len  │
│ ---            ┆ ---  │
│ str            ┆ u32  │
╞════════════════╪══════╡
│ Roads          ┆ 1733 │
│ Bylaw          ┆ 531  │
│ WRS            ┆ 281  │
│ CBS Inspection ┆ 240  │
│ Corporate      ┆ 169  │
└────────────────┴──────┘


service_name,len
str,u32
"""Roads - Snow and Ice Control""",1397




September 10th, 2014:
Total requests: 4621
shape: (5, 2)
┌────────────────┬──────┐
│ service_group  ┆ len  │
│ ---            ┆ ---  │
│ str            ┆ u32  │
╞════════════════╪══════╡
│ Parks          ┆ 2428 │
│ 311 Contact Us ┆ 965  │
│ Roads          ┆ 363  │
│ Bylaw          ┆ 263  │
│ CBS Inspection ┆ 153  │
└────────────────┴──────┘


service_name,len
str,u32
"""Parks - Tree Emergency - WAM""",2090




November 13th, 2018:
Total requests: 3405
shape: (5, 2)
┌────────────────┬──────┐
│ service_group  ┆ len  │
│ ---            ┆ ---  │
│ str            ┆ u32  │
╞════════════════╪══════╡
│ City Clerk's   ┆ 1512 │
│ CBS Inspection ┆ 293  │
│ Roads          ┆ 264  │
│ Finance        ┆ 241  │
│ Recreation     ┆ 226  │
└────────────────┴──────┘


service_name,len
str,u32
"""City Clerk's - Olympic VOTE 2018""",1512




April 3rd, 2020:
Total requests: 3812
shape: (5, 2)
┌───────────────┬──────┐
│ service_group ┆ len  │
│ ---           ┆ ---  │
│ str           ┆ u32  │
╞═══════════════╪══════╡
│ CFD           ┆ 2989 │
│ WRS           ┆ 186  │
│ Finance       ┆ 159  │
│ WATS          ┆ 81   │
│ Roads         ┆ 80   │
└───────────────┴──────┘


service_name,len
str,u32
"""CFD - Operation Birthdays""",2971


**The big spikes are real events.** Four days stand out from normal volume, and the service breakdown explains each one.

| Date | Requests | Main service | Cause |
|---|---|---|---|
| 2014-01-16 | 3,759 | Roads - Snow and Ice Control (1,397) | Snow and ice control surge. ([City of Calgary](https://newsroom.calgary.ca/calgary-road-closures---january-16-2014/)) |
| 2014-09-10 | 4,621 | Parks - Tree Emergency - WAM (2,090) | The September 2014 snowstorm, which damaged about half of Calgary's trees ([Calgary Herald](https://calgaryherald.com/news/local-news/september-snowstorm-damaged-about-half-of-all-calgary-trees)) |
| 2018-11-13 | 3,405 | City Clerk's - Olympic VOTE 2018 (1,512) | Plebiscite day for Calgary's 2026 Olympic bid ([Global News](https://globalnews.ca/news/4650203/calgary-olympic-plebiscite-results/)) |
| 2020-04-03 | 3,812 | CFD - Operation Birthdays (2,971) | The Calgary Fire Department's drive-by birthdays ([City of Calgary](https://newsroom.calgary.ca/calgary-fire-department-drive-by-birthdays/)) |


**Findings:**

- **The effectively starts on 2012-01-01**. 2010 has 39 requests and 2011 has 89, then about 470,000 to 550,000 per year from 2012 with no steep changes. Early January 2012 ramps up (194, 800, 1,438, 1,642 over the first four days).

- 604 of the 608 empty days are before 2012. The four after are 2012-03-10, 2014-12-31, 2017-01-21, and 2017-12-02.

- Volume is seasonal, peaking at 52,000 to 55,000 a month in June and July and bottoming near 30,000 in February. Weekdays carry 1.2 to 1.4 million requests each over the whole period, while Saturday carries 513,000 (about 660 on a typical Saturday) and Sunday 453,000.

- December 31st is a normal low day (14, 57, and 62 requests in other years), so a quiet 2013-12-31 is plausible. The three Saturdays above, plus **2018-10-27 (1 request against about 1,700 typical)** and **2020-03-28 (23)**, are far below a normal Saturday and look like source gaps or outages.

- The unusual-days list is otherwise Holidays: Christmas Day runs at 15% to 25% of normal.

- Three of the four are demand driven by something outside the city's control: two snow events and a vote. Each is dominated by a single service.

- The April 2020 spike is different: it is a fire department program creating requests in bulk, not residents calling in. The data includes requests that are not ordinary citizen calls.

- The two snow spikes are exactly what the weather join is for (checked below).

**Decision:** 

Start `dim_date` and the daily table at 2012-01-01. List the unexplainable gap days in the README.md. Use a robust method when correlating daily volume with weather.

## 10. Channel (source)

In [34]:
channel_share = (
    cleaned.group_by(pl.col("requested_date").dt.year().alias("year"), "source")
    .len()
    .with_columns(
        (pl.col("len") / pl.col("len").sum().over("year")).alias("share"),
        pl.date(pl.col("year"), 1, 1).alias("year"),
    )
    .sort("year")
)

display(channel_share.pivot(on="source", values="share", index="year"))

year,Web,Other,Phone,App
date,f64,f64,f64,f64
2010-01-01,0.102564,0.384615,0.512821,null
2011-01-01,0.089888,0.089888,0.820225,null
2012-01-01,0.09042,0.083509,0.826071,null
2013-01-01,0.093827,0.092249,0.797615,0.01631
2014-01-01,0.109626,0.093807,0.764621,0.031946
2015-01-01,0.107893,0.082872,0.758374,0.050862
2016-01-01,0.106175,0.096775,0.749435,0.047616
2017-01-01,0.110601,0.115431,0.714923,0.059046
2018-01-01,0.133439,0.121595,0.661013,0.083953


In [35]:
# Where exactly does the Web channel disappear?
disappearance = (
    cleaned.filter(
        pl.col("requested_date").is_between(date(2020, 6, 1), date(2021, 12, 31))
    )
    .group_by(pl.col("requested_date").dt.truncate("1mo").alias("month"), "source")
    .len()
    .with_columns((pl.col("len") / pl.col("len").sum().over("month")).alias("share"))
    .pivot(on="source", index="month", values="share")
    .sort("month")
)

display(disappearance)

month,Web,Phone,Other,App
date,f64,f64,f64,f64
2020-06-01,0.193024,0.54132,0.139522,0.126134
2020-07-01,0.169032,0.555871,0.146656,0.12844
2020-08-01,0.168645,0.546453,0.16006,0.124842
2020-09-01,0.158532,0.567071,0.170649,0.103748
2020-10-01,0.142823,0.586582,0.181468,0.089126
2020-11-01,0.170434,0.581992,0.161516,0.086058
2020-12-01,0.215862,0.509584,0.146232,0.128322
2021-01-01,0.171516,0.543175,0.175144,0.110165
2021-02-01,0.13908,0.573021,0.184088,0.103811


In [36]:
# What is "Other" made of in 2025?
cleaned.filter(
    (pl.col("requested_date").dt.year() == 2025) & (pl.col("source") == "Other")
).group_by("service_group").len().sort("len", descending=True).head(8)

service_group,len
str,u32
"""WRS""",77403
"""Roads""",59069
"""WATS""",38334
"""Bylaw""",36403
"""AT""",36136
"""Parks""",35155
"""CT""",15950
"""CFD""",11545


In [37]:
lines = (
    alt.Chart(channel_share)
    .mark_line(strokeWidth=3)
    .encode(
        x="year:T",
        y="share:Q",
        color=alt.Color(
            "source:N",
            scale=alt.Scale(
                domain=["Phone", "App", "Web", "Other"],
                range=["#7FA8D6", "#F4A261", "#76B893", "#E57373"],
            ),
        ),
        tooltip=["year:T", "source:N", alt.Tooltip("share:Q", format=".1%")],
    )
)

rule_df = pl.DataFrame({"year": ["2022-01-01"], "label": ["Web disappears"]})

rule = (
    alt.Chart(rule_df)
    .mark_rule(color="gray", strokeDash=[5, 5], strokeWidth=2)
    .encode(x="year:T")
)

label = (
    alt.Chart(rule_df)
    .mark_text(
        align="left",
        baseline="top",
        dx=6,
        dy=6,
        fontSize=13,
        color="gray",
    )
    .encode(
        x="year:T",
        y=alt.value(0),
        text="label:N",
    )
)

chart = (
    (lines + rule + label)
    .properties(width=1200, height=400)
    .configure_axis(labelFontSize=14, titleFontSize=16)
    .interactive()
)

chart

alt.LayerChart(...)

**Findings:** The channel field changed meaning, so the Phone, Web and Other split is not comparable across years.

| Year | Phone | Web | Other | App |
|---|---|---|---|---|
| 2012 | 83% | 9% | 8% | none |
| 2020 | 55% | 19% | 15% | 11% |
| 2021 | 54% | 3% | 29% | 14% |
| 2023 | 35% | 0.05% | 46% | 19% |
| 2024 | 9% | 0.05% | 69% | 22% |
| 2025 | none | 0.002% | 81% | 19% |

- **March to April 2021: Web became `Other`.** Web falls from 14% to 22% of requests to 7% in March 2021 and to about zero in April, while `Other` rises from 18% to 31%. Phone and App barely move. This is a sharp step, so a system or form change, not changing habits.

- **2022 to 2025: Phone fades into `Other`**, gradually, until Phone is zero. 311 is a phone service, so this is a recording change.

- In 2025, `Other` is spread across the large groups (WRS 77k, Roads 60k, WATS 38k, Bylaw 36k, AT 36k, Parks 36k), so it is the general default label and not one internal system.

- `App` grows smoothly from 1.6% in 2013 to about 22% in 2024, then holds near 19%. App versus everything else is the dependable split.

**Decision:** Add `channel_group` (App or Other channels) for the long-run view, and `is_channel_detail_reliable` (true before 2021-03-01) for the full split.

In [38]:
cleaned = cleaned.with_columns(
    pl.when(pl.col("source") == "App")
    .then(pl.lit("App"))
    .otherwise(pl.lit("Other channels"))
    .alias("channel_group"),
    (pl.col("requested_date") < date(2021, 3, 1)).alias("is_channel_detail_reliable"),
)

## 11. Dimension consistency

The star schema keys on community code and service name: check to see if these are one-to-one.

In [39]:
pairs = cleaned.filter(
    pl.col("comm_code").is_not_null() & pl.col("comm_name").is_not_null()
)

# Codes with more than one name, and names with more than one code. Expect no rows.
print(
    pairs.group_by("comm_code")
    .agg(pl.col("comm_name").unique().alias("names"))
    .filter(pl.col("names").list.len() > 1)
)
print(
    pairs.group_by("comm_name")
    .agg(pl.col("comm_code").unique().alias("codes"))
    .filter(pl.col("codes").list.len() > 1)
)

shape: (0, 2)
┌───────────┬───────────┐
│ comm_code ┆ names     │
│ ---       ┆ ---       │
│ str       ┆ list[str] │
╞═══════════╪═══════════╡
└───────────┴───────────┘
shape: (0, 2)
┌───────────┬───────────┐
│ comm_name ┆ codes     │
│ ---       ┆ ---       │
│ str       ┆ list[str] │
╞═══════════╪═══════════╡
└───────────┴───────────┘


In [40]:
# Services handled by more than one agency since 2024
(
    cleaned.filter(
        (pl.col("requested_date") >= date(2024, 1, 1))
        & pl.col("agency_responsible").is_not_null()
    )
    .group_by("service_name")
    .agg(
        pl.col("agency_responsible").unique().alias("agencies"),
        pl.len().alias("requests"),
    )
    .filter(pl.col("agencies").list.len() > 1)
    .sort("requests", descending=True)
)

service_name,agencies,requests
str,list[str],u32
"""Green Line Inquiry""","[""IS - Green Line LRT - SE Project"", ""TRAN - Green Line""]",311
"""BIA Requests""","[""CS - Partnerships"", ""CS - Community Strategies""]",241
"""CAI - City Online - Feedback Form Only""","[""PICS - Information Technology"", ""PICS - Collaboration, Analytics and Innovation""]",33
"""CHO - Affordable Housing""","[""CS - Chief Housing Office"", ""CEO - Chief Housing Office""]",27


**Findings:**

- Each `comm_code` has exactly one `comm_name` and the reverse, so `dim_community` can key on the code. The one extra distinct name (324 names against 323 codes) is probably a single request that has a name and no code.

- Since 2024 only four services have two agencies, covering 612 requests: `Green Line Inquiry`, `BIA Requests`, `CAI - City Online - Feedback Form Only`, and `CHO - Affordable Housing`. All four are renames or reorganizations. Earlier years have many more.

## 12. IDs, nulls, and time to close

In [41]:
# IDs should look like YY-NNNNNNNN, with the year matching the request year
id_year = pl.col("service_request_id").str.slice(0, 2).cast(pl.Int32, strict=False)
cleaned.filter(
    ~pl.col("service_request_id").str.contains(r"^\d{2}-\d{8}$")
    | (id_year != pl.col("requested_date").dt.year() % 100)
)

service_request_id,requested_date,updated_date,closed_date,status_description,source,service_name,agency_responsible,comm_code,comm_name,location_type,longitude,latitude,_ingested_at,is_legacy_service,service_group,is_unnamed_area,is_bulk_closure,channel_group,is_channel_detail_reliable
str,date,date,date,str,str,str,str,str,str,str,f64,f64,str,bool,str,bool,bool,str,bool
"""13-00221829""",2012-05-14,2013-05-14,2013-05-14,"""Closed""","""Other""","""WRS - Waste DP - Review and Inquiries""","""UEP - Waste and Recycling Services""",null,null,null,null,null,"""2026-11-13T02:10:09.894542+00:00""",false,"""WRS""",false,false,"""Other channels""",true


In [42]:
cleaned.null_count().transpose(
    include_header=True, header_name="column", column_names=["nulls"]
)

column,nulls
str,u32
"""service_request_id""",0
"""requested_date""",0
"""updated_date""",77829
"""closed_date""",76751
"""status_description""",0
"""source""",0
"""service_name""",8721
"""agency_responsible""",312
"""comm_code""",436103


In [43]:
# Time to close by request year, without bulk closures
days = (pl.col("closed_date") - pl.col("requested_date")).dt.total_days()
(
    cleaned.filter(pl.col("closed_date").is_not_null() & ~pl.col("is_bulk_closure"))
    .group_by(pl.col("requested_date").dt.year().alias("year"))
    .agg(
        days.median().alias("median_days"),
        days.quantile(0.9).alias("p90_days"),
        pl.len().alias("closed"),
    )
    .sort("year")
)

year,median_days,p90_days,closed
i32,f64,f64,u32
2010,4404.0,4622.0,39
2011,4017.0,4186.0,89
2012,1.0,16.0,466797
2013,1.0,16.0,507062
2014,1.0,18.0,547893
2015,1.0,23.0,496346
2016,2.0,22.0,488011
2017,2.0,22.0,515707
2018,2.0,33.0,535483


**Findings:**

- Every ID matches the format. One request, `13-00221829`, has a `13` prefix but a request date in 2012; it closed exactly a year after that date. Likely a mistyped year or a late-numbered request, which is one row in 7.5 million.

- The null profile matches the earlier findings: `service_name` and `service_group` 8,721, `agency_responsible` 312, `closed_date` 76,751, `updated_date` 77,829, `comm_code` and `comm_name` about 436,000, `location_type` 435,287, latitude and longitude 436,167. `requested_date`, `status_description`, `source` and `_ingested_at` are never null.

- The 2010 and 2011 medians are about 4,000 days, on only 128 requests in total: those old requests were closed years later (several on the same 2022 dates). From 2012 the median is 1 to 4 days and the 90th percentile 16 to 33 days. The median drifts from 1 (2012 to 2015) to about 3 in recent years; this could be slower service or a different mix of requests.

- **The most recent years look faster than they are.** Requests from 2025 and 2026 that are still open are not counted, so compare complete years only.

## 13. Weather

Daily weather for Calgary International Airport (ECCC `climate-daily`, station 3031092). Like the 311 data, the raw layer holds serveral versions of a day because the last 14 days are refetched.

In [44]:
weather = (
    pl.scan_parquet(RAW_WEATHER)
    .sort("_ingested_at", descending=True)
    .unique(subset="local_date", keep="first", maintain_order=True)
    .sort("local_date")
    .collect()
)
print(
    f"{weather.height:,} days, {weather['local_date'].min()} to {weather['local_date'].max()}"
)

MEASURES = [
    "mean_temperature",
    "min_temperature",
    "max_temperature",
    "total_precipitation",
    "total_rain",
    "total_snow",
    "snow_on_ground",
    "direction_max_gust",
    "speed_max_gust",
    "cooling_degree_days",
    "heating_degree_days",
    "min_rel_humidity",
]

5,237 days, 2012-07-12 to 2026-11-12


In [45]:
# Days with no row at all
full = pl.date_range(
    weather["local_date"].min(), weather["local_date"].max(), interval="1d", eager=True
)
print(
    pl.DataFrame({"local_date": full}).join(
        weather.select("local_date"), on="local_date", how="anti"
    )
)

# Days with a row but every measurement null
print(
    "empty days:", weather.filter(pl.all_horizontal(pl.col(MEASURES).is_null())).height
)

shape: (0, 1)
┌────────────┐
│ local_date │
│ ---        │
│ date       │
╞════════════╡
└────────────┘
empty days: 91


In [46]:
weather.group_by(pl.col("local_date").dt.year().alias("year")).agg(
    pl.col(MEASURES).is_null().mean()
).sort("year")

year,mean_temperature,min_temperature,max_temperature,total_precipitation,total_rain,total_snow,snow_on_ground,direction_max_gust,speed_max_gust,cooling_degree_days,heating_degree_days,min_rel_humidity
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2012,0.16763,0.16763,0.16763,0.17341,0.184971,0.17341,0.653179,0.17341,0.17341,0.16763,0.16763,1.0
2013,0.164384,0.161644,0.164384,0.164384,0.169863,0.169863,0.663014,0.169863,0.169863,0.164384,0.164384,1.0
2014,0.0,0.0,0.0,0.016438,0.021918,0.005479,0.517808,0.008219,0.0,0.0,0.0,1.0
2015,0.0,0.0,0.0,0.013699,0.013699,0.00274,0.632877,0.0,0.0,0.0,0.0,1.0
2016,0.005464,0.002732,0.002732,0.008197,0.019126,0.016393,0.702186,0.013661,0.013661,0.005464,0.005464,1.0
2017,0.049315,0.041096,0.008219,0.005479,0.043836,0.010959,0.624658,0.010959,0.010959,0.049315,0.049315,1.0
2018,0.024658,0.024658,0.005479,0.00274,0.041096,0.010959,0.506849,0.063014,0.073973,0.024658,0.024658,0.832877
2019,0.019178,0.019178,0.019178,0.005479,0.068493,0.065753,0.542466,0.194521,0.257534,0.019178,0.019178,0.013699
2020,0.021858,0.021858,0.021858,0.002732,0.046448,0.046448,0.551913,0.117486,0.144809,0.021858,0.021858,0.090164


In [47]:
# Impossible values. All should be 0.
weather.select(
    (pl.col("min_temperature") > pl.col("mean_temperature"))
    .sum()
    .alias("min_above_mean"),
    (pl.col("mean_temperature") > pl.col("max_temperature"))
    .sum()
    .alias("mean_above_max"),
    (pl.col("min_temperature") < -50).sum().alias("below_minus_50"),
    (pl.col("max_temperature") > 45).sum().alias("above_45"),
    (pl.col("total_precipitation") < 0).sum().alias("negative_precipitation"),
)

min_above_mean,mean_above_max,below_minus_50,above_45,negative_precipitation
u32,u32,u32,u32,u32
0,0,0,0,0


In [48]:
# How many 311 requests land on a day with no usable weather?
requests_per_day = (
    cleaned.group_by("requested_date")
    .len()
    .rename({"requested_date": "local_date", "len": "requests"})
)
joined = requests_per_day.join(
    weather.select("local_date", pl.lit(True).alias("has_row"), "mean_temperature"),
    on="local_date",
    how="left",
)
joined.select(
    pl.col("requests").sum().alias("requests"),
    pl.col("requests")
    .filter(pl.col("has_row").is_null())
    .sum()
    .alias("no_weather_row"),
    pl.col("requests")
    .filter(pl.col("has_row").is_not_null() & pl.col("mean_temperature").is_null())
    .sum()
    .alias("no_temperature"),
)

requests,no_weather_row,no_temperature
u32,u32,u32
7554152,248402,273311


In [49]:
# Could a missing mean be rebuilt from min and max? (It cannot: this is 0.)
weather.filter(
    pl.col("mean_temperature").is_null()
    & pl.col("min_temperature").is_not_null()
    & pl.col("max_temperature").is_not_null()
).height

0

**Findings:**

- No missing dates: every day from 2012-07-12 to the latest has a row. 91 days have a row with every measurement null, about 89 of them in 2012 and 2013 (roughly 17% of days in those years).

- About 190 days in total lack a mean temperature, so temperature gaps are wider than the 91 empty days suggest. 2014 to 2016 are nearly complete and later years have a few percent missing. The days missing a mean also lack a minimum, so it cannot be rebuilt.

- **Humidity is unusable before 2018**: 100% null for 2012 to 2017 and 83% null in 2018.

- Gust speed and direction jump from about 1% null to about 15% to 25% null from 2019. A null may mean no gust above the reporting threshold, so do not treat it as zero or average across the change.

- `snow_on_ground` is 50% to 70% null every year, which fits a seasonal measurement. Null is not zero.

- No impossible values (min above mean, below -50, above 45, negative precipitation).

- Of 7,554,152 requests, **248,402 (3.3%)** fall before the weather starts and **273,311 (3.6%)** fall on days with no mean temperature: about **6.9%** have no temperature.

**Decisions:** 

Do not impute missing weather. Add `has_temperature` beside `has_observations`. Use 2014 onward for temperature and precipitation and 2019 onward for humidity.

## 13. Decisions for dbt

The rules to put into `stg_service_requests` and the marts. After the staging model is built, its row count should equal the notebook's:

In [50]:
print(
    f"{cleaned.height:,} requests in the cleaned frame (the dbt staging model should match)"
)

7,554,152 requests in the cleaned frame (the dbt staging model should match)


**Staging rules**
1. One row per request: order by `updated_date` descending (nulls last), then `_ingested_at` descending, and keep the first.

2. Drop `TO BE DELETED`.

3. Trim text, collapse spaces, and turn `N/A`, `null`, `None` and blanks into null (service, agency, community, location type).

4. Service names: flag `is_legacy_service`, strip the `z` marker, apply the canonical-name seed (the `name_map` table above), then compute `service_group`.

5. Communities: normalize slash spacing; flag `is_unnamed_area`.

6. Cast the three dates to `date`.

7. Add `channel_group` and `is_channel_detail_reliable` (before 2021-03-01).

8. Add `is_bulk_closure` (batches of at least 1,000 closures with a median age of at least 100 days, requests open 100 days or more). Keep `days_to_close` and the status-versus-date conflict flag.

**Star schema**
- `dim_service` on the cleaned service name (group and legacy flag), a separate `dim_agency`, and `dim_community` with one median coordinate per community (null for unnamed areas). The fact table carries the keys.

- `dim_date` and the daily table start at 2012-01-01. Report medians for time to close.

**Tests**: unique and not null on the request ID; accepted values for status and source; warning-level tests for status versus closed date, requests without coordinates, and ID year versus request year.

**Limits to state in the README**

- Locations are community centrepoints, not addresses.

- The channel field changed meaning in 2021 and again from 2022 to 2025.

- Roughly 6.9% of requests have no temperature; humidity is unusable before 2018; gust and snow-on-ground nulls are not zeros.

- Data is complete from 2012; a handful of Saturdays look like source gaps; some requests are created in bulk by programs; some backlog cleanups close old requests in one day.